# 🔧 ScrapScan — Training Notebook
**IIM MATRIXx 2026 | IIT ISM Dhanbad**

This notebook trains a multi-task EfficientNet-B0 to classify ferrous scrap and detect zinc contamination.

**Runtime:** Set to T4 GPU (Runtime → Change runtime type → T4 GPU)

**Total time:** ~2–3 hours depending on dataset size

In [ ]:
# CELL 1 — Install dependencies
!pip install -q torch torchvision timm albumentations icrawler yt-dlp opencv-python-headless onnx onnxruntime onnxscript pillow scikit-learn matplotlib seaborn
print('✅ Dependencies installed')

In [ ]:
# CELL 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os
SAVE_DIR = '/content/drive/MyDrive/scrapscan_training'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Save directory: {SAVE_DIR}')

In [ ]:
# CELL 3 — Image scraping via Bing
import os
from icrawler.builtin import BingImageCrawler

SEARCH_QUERIES = {
    'hms1': ['heavy melting steel scrap pile truck', 'HMS1 structural steel scrap yard India',
             'thick steel beam scrap metal recycling', 'iron girder scrap pile'],
    'hms2': ['light steel scrap sheet metal pile', 'thin sheet metal stampings scrap yard',
             'HMS2 steel scrap recycling', 'pressed steel scrap automobile parts'],
    'galv': ['galvanized steel scrap pile recycling', 'zinc coated steel scrap metal',
             'galvanized sheet metal spangle surface', 'corrugated galvanized iron scrap'],
    'ss':   ['stainless steel scrap pile 304', 'shiny stainless steel metal recycling',
             'SS scrap yard bright metal'],
    'nonfe':['copper scrap metal pile reddish', 'aluminium scrap metal recycling bright',
             'mixed non ferrous scrap yard copper aluminium'],
    'mixed':['contaminated scrap metal plastic mixed', 'mixed scrap pile rubber plastic metal junk',
             'scrap yard contamination waste mixed materials']
}

for class_name, queries in SEARCH_QUERIES.items():
    for i, query in enumerate(queries):
        out_dir = f'/content/data/raw/{class_name}/q{i}'
        os.makedirs(out_dir, exist_ok=True)
        print(f'Scraping [{class_name}] "{query}"')
        try:
            crawler = BingImageCrawler(storage={'root_dir': out_dir},
                                       feeder_threads=1, parser_threads=1, downloader_threads=4)
            crawler.crawl(keyword=query, max_num=200, min_size=(100,100))
        except Exception as e:
            print(f'  Skipped: {e}')

print('\n=== Scraping done ===')
for cls in SEARCH_QUERIES:
    n = sum(len(f) for _,_,f in os.walk(f'/content/data/raw/{cls}'))
    print(f'  {cls}: {n} images')

In [ ]:
# CELL 4 — YouTube frame extraction (galvanized steel)
from google.colab import files
uploaded = files.upload()   # Upload the youtube_urls.txt file

import yt_dlp, cv2, os

def extract_youtube_frames(url, output_dir, every_n=45):
    ydl_opts = {'format': 'best[height<=480]', 'outtmpl': '/content/temp_vid.mp4', 'quiet': True}
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
        cap = cv2.VideoCapture('/content/temp_vid.mp4')
        os.makedirs(output_dir, exist_ok=True)
        frame_count, saved = 0, 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            if frame_count % every_n == 0:
                cv2.imwrite(f'{output_dir}/yt_{saved:04d}.jpg', frame)
                saved += 1
            frame_count += 1
        cap.release()
        if os.path.exists('/content/temp_vid.mp4'):
            os.remove('/content/temp_vid.mp4')
        print(f'Extracted {saved} frames from {url}')
    except Exception as e:
        print(f'Failed {url}: {e}')

try:
    with open('/content/youtube_urls.txt') as f:
        urls = [l.strip() for l in f if l.strip() and not l.startswith('#')]
    print(f'Found {len(urls)} YouTube URLs')
    for url in urls:
        extract_youtube_frames(url, '/content/data/raw/galv/youtube')
except FileNotFoundError:
    print('WARNING: youtube_urls.txt not found.')
    print('Upload ml/config/youtube_urls.txt to Colab to enable this step.')

In [ ]:
# CELL 5 — Image counts + MANUAL CURATION PROMPT
import os
CLASSES = ['hms1','hms2','galv','ss','nonfe','mixed']
print('=== CLASS IMAGE COUNTS ===')
for cls in CLASSES:
    n = sum(len(f) for _,_,f in os.walk(f'/content/data/raw/{cls}'))
    status = '✅' if n >= 200 else '⚠️ LOW'
    print(f'  {cls:8s}: {n:5d} images  {status}')

print('''
╔══════════════════════════════════════════════════════════════════╗
║  MANUAL STEP REQUIRED — READ BEFORE CONTINUING                  ║
╠══════════════════════════════════════════════════════════════════╣
║  Review images in /content/data/raw/<class>/ using Colab Files. ║
║  Delete images that are clearly wrong (e.g. cars in hms1).      ║
║  Spend 20–30 minutes — this is the most impactful step.         ║
║  Target: ≥200 clean images per class before splitting.          ║
╚══════════════════════════════════════════════════════════════════╝
''')

In [ ]:
# CELL 6 — Dataset split (70/20/10 train/val/test)
import os, shutil, random
random.seed(42)

def split_dataset(raw_dir, split_dir, ratios=(0.7, 0.2, 0.1)):
    splits = ['train', 'val', 'test']
    EXTS = {'.jpg','.jpeg','.png','.webp','.bmp'}
    for cls in os.listdir(raw_dir):
        cls_path = os.path.join(raw_dir, cls)
        if not os.path.isdir(cls_path): continue
        imgs = []
        for root, _, files in os.walk(cls_path):
            imgs += [os.path.join(root, f) for f in files if os.path.splitext(f)[1].lower() in EXTS]
        random.shuffle(imgs)
        n = len(imgs)
        n_train = int(n * ratios[0])
        n_val   = int(n * ratios[1])
        chunks  = [imgs[:n_train], imgs[n_train:n_train+n_val], imgs[n_train+n_val:]]
        for split, chunk in zip(splits, chunks):
            dst = os.path.join(split_dir, split, cls)
            os.makedirs(dst, exist_ok=True)
            for src in chunk:
                shutil.copy2(src, os.path.join(dst, os.path.basename(src)))
        print(f'  {cls}: {n_train} train / {n_val} val / {len(chunks[2])} test')

split_dataset('/content/data/raw', '/content/data/split')
print('\nDataset split complete!')

In [ ]:
# CELL 7 — Model + Dataset classes
import torch
import torch.nn as nn
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
import numpy as np
from torch.utils.data import Dataset
import os

CLASS_KEYS = ['hms1','hms2','galv','ss','nonfe','mixed']
ZINC_CLASSES = {'galv'}

class ScrapScanModel(nn.Module):
    def __init__(self, num_classes=6, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b0', pretrained=pretrained, num_classes=0, global_pool='avg')
        feat_dim = self.backbone.num_features
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Sequential(nn.Linear(feat_dim,256), nn.ReLU(True), nn.Dropout(0.2), nn.Linear(256, num_classes))
        self.zinc_detector = nn.Sequential(nn.Linear(feat_dim,128), nn.ReLU(True), nn.Dropout(0.2), nn.Linear(128,1))
    def forward(self, x):
        f = self.dropout(self.backbone(x))
        return self.classifier(f), self.zinc_detector(f)
    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad=False
    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad=True

class ScrapDataset(Dataset):
    def __init__(self, root, transform=None):
        self.transform = transform
        self.samples = []
        self.class_to_idx = {k:i for i,k in enumerate(CLASS_KEYS)}
        for cls in CLASS_KEYS:
            d = os.path.join(root, cls)
            if not os.path.exists(d): continue
            zinc = 1.0 if cls in ZINC_CLASSES else 0.0
            for f in os.listdir(d):
                if f.lower().endswith(('.jpg','.jpeg','.png','.webp','.bmp')):
                    self.samples.append((os.path.join(d,f), self.class_to_idx[cls], zinc))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, lbl, zinc = self.samples[idx]
        try: img = np.array(Image.open(path).convert('RGB'))
        except: img = np.zeros((224,224,3), dtype=np.uint8)
        if self.transform:
            img = self.transform(image=img)['image']
        return {'image': img, 'label': torch.tensor(lbl,dtype=torch.long), 'zinc': torch.tensor(zinc,dtype=torch.float32)}

train_transform = A.Compose([
    A.RandomResizedCrop(size=(224, 224), scale=(0.6, 1.0)),
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.3), A.RandomRotate90(p=0.5),
    A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05, p=0.8), A.GaussianBlur(blur_limit=(3, 7), p=0.3),
    A.GaussNoise(p=0.3),
    A.RandomShadow(p=0.3), A.CLAHE(p=0.2),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), ToTensorV2()
])
val_transform = A.Compose([
    A.Resize(height=224, width=224), A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), ToTensorV2()
])
print('✅ Model and Dataset classes defined')

In [ ]:
# CELL 8 — Training setup
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

train_ds = ScrapDataset('/content/data/split/train', transform=train_transform)
val_ds   = ScrapDataset('/content/data/split/val',   transform=val_transform)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

model = ScrapScanModel(num_classes=6, pretrained=True).to(device)

# Class weights
counts = torch.zeros(6)
for _,lbl,_ in train_ds.samples: counts[lbl] += 1
class_weights = (counts.sum() / (6 * counts)).to(device)
print(f'Class weights: {class_weights.tolist()}')

cls_crit  = nn.CrossEntropyLoss(weight=class_weights)
zinc_crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([5.0]).to(device))

os.makedirs('/content/checkpoints', exist_ok=True)
best_acc = 0.0

def run_epoch(model, dl, opt=None, train=True):
    model.train() if train else model.eval()
    tot_loss, correct, total = 0., 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for b in dl:
            imgs, lbls, zinc = b['image'].to(device), b['label'].to(device), b['zinc'].to(device)
            cls_l, zinc_l = model(imgs)
            loss = cls_crit(cls_l, lbls) + 0.4 * zinc_crit(zinc_l.squeeze(1), zinc)
            if train:
                opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            tot_loss += loss.item()*imgs.size(0)
            correct  += (cls_l.argmax(1)==lbls).sum().item()
            total    += imgs.size(0)
    return tot_loss/total, correct/total

print('✅ Training setup complete')

In [ ]:
# CELL 9 — STAGE 1: Train heads only (10 epochs)
print('=== STAGE 1: Heads only ===')
model.freeze_backbone()
opt1 = AdamW(filter(lambda p:p.requires_grad, model.parameters()), lr=3e-4, weight_decay=1e-4)
sch1 = CosineAnnealingLR(opt1, T_max=10)

for ep in range(1, 11):
    t0 = time.time()
    tl, ta = run_epoch(model, train_dl, opt1, train=True)
    vl, va = run_epoch(model, val_dl,   None,  train=False)
    sch1.step()
    print(f'[S1 E{ep:02d}] loss={tl:.4f}/{vl:.4f}  acc={ta:.3f}/{va:.3f}  ({time.time()-t0:.0f}s)')
    if va > best_acc:
        best_acc = va
        torch.save(model.state_dict(), '/content/checkpoints/best.pt')
        print(f'  ✅ Best val_acc={best_acc:.4f}')

In [ ]:
# CELL 10 — STAGE 2: Full fine-tuning (30 epochs)
print('=== STAGE 2: Full fine-tuning ===')
model.unfreeze_backbone()
opt2 = AdamW([
    {'params': model.backbone.parameters(),     'lr': 5e-5},
    {'params': model.classifier.parameters(),   'lr': 2e-4},
    {'params': model.zinc_detector.parameters(),'lr': 2e-4},
], weight_decay=1e-4)
sch2 = CosineAnnealingLR(opt2, T_max=30)

for ep in range(1, 31):
    t0 = time.time()
    tl, ta = run_epoch(model, train_dl, opt2, train=True)
    vl, va = run_epoch(model, val_dl,   None,  train=False)
    sch2.step()
    print(f'[S2 E{ep:02d}] loss={tl:.4f}/{vl:.4f}  acc={ta:.3f}/{va:.3f}  ({time.time()-t0:.0f}s)')
    if va > best_acc:
        best_acc = va
        torch.save(model.state_dict(), '/content/checkpoints/best.pt')
        torch.save(model.state_dict(), f'{SAVE_DIR}/best.pt')
        print(f'  ✅ Best val_acc={best_acc:.4f}')

print(f'\nTraining complete! Best val_acc = {best_acc:.4f}')

In [ ]:
# CELL 11 — Evaluate + Confusion Matrix
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

model.load_state_dict(torch.load('/content/checkpoints/best.pt', map_location=device))
model.eval()
test_ds = ScrapDataset('/content/data/split/test', transform=val_transform)
test_dl = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

all_p, all_l = [], []
with torch.no_grad():
    for b in test_dl:
        imgs = b['image'].to(device)
        out,_ = model(imgs)
        all_p.extend(out.argmax(1).cpu().numpy())
        all_l.extend(b['label'].numpy())

print(classification_report(all_l, all_p, target_names=CLASS_KEYS))

cm = confusion_matrix(all_l, all_p)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', xticklabels=CLASS_KEYS, yticklabels=CLASS_KEYS)
plt.title('ScrapScan Confusion Matrix'); plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# CELL 12 — ONNX Export
import torch, onnx, onnxruntime as ort, shutil

model.load_state_dict(torch.load('/content/checkpoints/best.pt', map_location='cpu'))
model.eval().cpu()

dummy = torch.randn(1,3,224,224)
torch.onnx.export(
    model, dummy, '/content/scrapscan_model.onnx',
    export_params=True, opset_version=11, do_constant_folding=True,
    input_names=['input'], output_names=['class_logits','zinc_logit'],
    dynamic_axes={'input':{0:'batch_size'}}
)

m = onnx.load('/content/scrapscan_model.onnx')
onnx.checker.check_model(m)
print('✅ ONNX model verified')

sess = ort.InferenceSession('/content/scrapscan_model.onnx')
out  = sess.run(None, {'input': dummy.numpy()})
print(f'ORT test OK — class_logits: {out[0].shape}, zinc_logit: {out[1].shape}')

shutil.copy('/content/scrapscan_model.onnx', f'{SAVE_DIR}/scrapscan_model.onnx')
print(f'Saved to Drive: {SAVE_DIR}/scrapscan_model.onnx')

In [ ]:
# CELL 13 — Download model
from google.colab import files
files.download('/content/scrapscan_model.onnx')

print('''
╔════════════════════════════════════════════════════════╗
║  NEXT STEPS                                            ║
╠════════════════════════════════════════════════════════╣
║  1. Wait for scrapscan_model.onnx to download          ║
║  2. Copy it to your project: public/models/            ║
║  3. Run: npm start                                     ║
║  4. The app will switch to Production Mode             ║
║     (green dot in header, no Demo banner)              ║
╚════════════════════════════════════════════════════════╝
''')